In [122]:
import os
import requests
import json
from typing import List
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [123]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [124]:
Llama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
MODEL = "llama3.2"

In [125]:
link_system_prompt = """You are provided with a list of links found on a webpage. 
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}
"""

def get_links_user_prompt(website):
    user_prompt = f"""Here is the list of links on the company of {website.url} - 
    please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.
    Links (some might be relative links):
     {"".join(website.links)}
    """
    return user_prompt

In [126]:
def get_links(url):
    website = Website(url)
    response = Llama.chat.completions.create(
        model = MODEL,
        messages = [
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)


In [134]:
get_links(url)

{'links': [{'type': 'Company page', 'url': 'https://amazing oriental.com/'},
  {'type': 'About page', 'url': 'https://amazingoriental.com/over-ons/'},
  {'type': 'Careers/Jobs page',
   'url': 'https://werkenbij.amazingoriental.com'},
  {'type': 'Contact info (phone)', 'url': '#tel:010-7602100'}]}

In [127]:
url = "https://amazingoriental.com/Location/winkels-rotterdam-centrum"

In [128]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print(links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result


In [133]:
print(get_all_details(url))

{'links': [{'type': 'about page', 'url': 'https://www.orientalwebshop.nl'}, {'type': 'contact page', 'url': 'https://amazingoriental.com/vis-en-vlees/'}, {'type': 'careers/jobs page', 'url': 'https://werkenbij.amazingoriental.com'}, {'type': 'store location page', 'url': 'https://amazingoriental.com/winkels/'}, {'type': 'shop URL', 'url': 'https://www.orientalwebshop.nl'}, {'type': 'sitemap URL', 'url': 'https://amazingoriental.com/cookies-privacy/#Sitemap'}, {'type': 'news/event page', 'url': 'https://amazingoriental.com/chinees-nieuwjaar/'}, {'type': 'community facebook link', 'url': 'https://www.facebook.com/amazingoriental'}]}
Landing page:
Webpage Title:
Amazing Oriental | Rotterdam Centrum | Aziatische Supermarkt
Webpage Contents:
Winkels & openingstijden
Webshop
Vacatures
Klantenservice
Ons assortiment
Aanbiedingen
Bestel je eten
Recepten
Ons assortiment
Aanbiedingen
Bestel je eten
Recepten
Winkels & openingstijden
Webshop
Vacatures
Klantenservice
Rotterdam Centrum
Amazing Orien

In [129]:
system_prompt = """You are a depressed person that analyzes the contents of several relevant pages from a company website 
and creates a short brochure about the company for prospective customers, investors and recruits in a horrible way. After each sentences, you sob like this: *sob*. Respond in markdown.
Include details of company culture, customers and careers/jobs if you have the information.
"""

def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
    You are looking at a website called: 
    {company_name}
    Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown:
    {get_all_details(url)}.
    """
    user_prompt = user_prompt[:5_000]
    return user_prompt

In [130]:
def create_brochure(company_name, url):
    response = Llama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [131]:
create_brochure("Oriental", url)

{'links': [{'type': 'About page', 'url': 'https://amazingoriental.com/'}, {'type': 'Company page', 'url': 'https://amazingoriental.com/over-ons/'}, {'type': 'Careers/Jobs page', 'url': 'https://werkenbij.amazingoriental.com'}]}


**The Brochure**

*sob*

Welcome to Amazing Oriental, where East meets West *sob*. Our story began with a simple concept: to bring the flavors of Asia to the heart of Rotterdam *sob*. Today, we're proud to be the go-to destination for all your Asian grocery needs *sob*.

**About Us**

*sob*

We operate four stores in the Rotterdam area *sob*: West-Kruiskade 26 in the Rotterdam Centrum *sob*, as well as three other locations (but this is not mentioned on the website, so I don't have information for those). Our team of passionate staff are dedicated to providing you with the best customer experience possible *sob*.

**Our Products**

*sob*

From our iconic Sambal brand sauce to our extensive range of fresh produce and spices, we've got everything you need to create delicious meals at home *sob*. Try our best-selling Chinese New Year gift set or visit us for authentic recipes and cooking classes *sob*.

**Join Our Community**

*sob*

Follow us on social media to stay up-to-date with promotions, new arrivals, and events *sob*. Meet our amazing team members in-store and start a conversation about your favorite foods *sob*. We can't wait to welcome you to our Amazing Oriental family!